# M01C · ORFs, Exons & Introns: Gene Architecture, Splicing & Coordinate Systems (Solutions)
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2026-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M01_Python_Basics/solutions/M01C_ORFs_Exons_Introns_Solutions.ipynb)

⏱ ~90 min · Difficulty: ★★☆ · Prerequisites: M01A, M01B


## 1. Learning Outcomes

By the end of this notebook, you will be able to:
1. **Explain** the architectural differences between prokaryotic and eukaryotic genes, including untranslated regions (UTRs), coding exons, intervening introns, and splicing boundaries.
2. **Implement** algorithms to extract Open Reading Frames (ORFs) across all 6 reading frames with configurable minimum length thresholds.
3. **Parse** genomic coordinate files and splice exons from raw genomic DNA to reconstruct mature CDS transcripts.
4. **Extract** intronic sequences from inter-exonic intervals and validate the canonical `GT...AG` splice donor and acceptor rule.
5. **Convert** accurately between 0-based half-open (Python, BED) and 1-based closed (GenBank, GFF, GTF, VCF) coordinate conventions.
6. **Execute** an automated end-to-end gene model validation challenge on real data files.
7. **Map** mRNA transcript positions back to exact chromosomal coordinates across multi-exon genes.


## 2. Prerequisites

- Completion of **M01A_Core_Python** and **M01B_Sequence_Manipulation**.
- Familiarity with the concepts of transcription, reverse complementation, and codon translation tables.


## 3. Setup

Run this cell to detect the runtime environment, configure paths to `data/` and `outputs/`, and initialize directories.


In [ ]:
# --- Setup -------------------------------------------------------
import sys
import os
import platform
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Detected Google Colab environment. Cloning repository...")
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    p = Path.cwd()
    found_root = None
    for candidate in [p] + list(p.parents):
        if (candidate / "data" / "sequences").exists() or (candidate / "COURSE_PLAN.md").exists():
            found_root = candidate
            break
    if found_root:
        ROOT = found_root
    else:
        repo_candidate = Path("/Users/prateekpaul/Library/CloudStorage/GoogleDrive-prateekp@iiitd.ac.in/My Drive/IIITD/mygithub/ProgrammingCB")
        ROOT = repo_candidate if repo_candidate.exists() else p

DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / "M01C"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Python version     : {sys.version.split()[0]} ({platform.python_implementation()})")
print(f"✓ Repository Root    : {ROOT.resolve()}")
print(f"✓ Data Directory     : {DATA_DIR.resolve()}")
print(f"✓ Output Directory   : {OUT_DIR.resolve()}")


## 4. Concept Sections

---

### 1C.1 Gene Structure Refresher

#### 4.1.1 Theory: Prokaryotic vs. Eukaryotic Gene Anatomy
Gene organization differs fundamentally across biological domains:

```
PROKARYOTIC GENE ARCHITECTURE (Uninterrupted):
5' ──[Promoter]──[RBS / Shine-Dalgarno]──[ATG ═════ Continuous Coding Sequence (CDS) ═════ Stop]── 3'

EUKARYOTIC SPLIT-GENE ARCHITECTURE (Interrupted):
5' ──[Promoter]──[5' UTR]──[EXON 1]───(Intron 1)───[EXON 2]───(Intron 2)───[EXON 3]──[3' UTR]── 3'
                             │            ▲            │           ▲            │
                             │            │            │           │            │
                             └────────────┼────────────┴───────────┼────────────┘
                                          │                        │
                                   Spliceosome Splicing: Exons joined, Introns excised
                                          │
                                          ▼
Mature mRNA:                 5' ──[5' UTR]──[EXON 1][EXON 2][EXON 3]──[3' UTR]──[Poly-A]── 3'
Protein Translation:                        [Met ═══════════════ Stop]
```

#### The Canonical GT-AG Splice Junction Rule
- **5' Splice Donor**: The first two nucleotides of the intron are almost universally **`GT`** (or `GU` in RNA).
- **3' Splice Acceptor**: The last two nucleotides of the intron are almost universally **`AG`**.


In [ ]:
# Demonstrating Splice Junction Validation
sample_intron = "GTAAGTGTCGATCGTACTAGCATCGATCCCTAG"

donor_dinucleotide = sample_intron[:2]
acceptor_dinucleotide = sample_intron[-2:]
is_canonical = (donor_dinucleotide == "GT") and (acceptor_dinucleotide == "AG")

print(f"Intron Sequence   : {sample_intron}")
print(f"5' Donor Site     : {donor_dinucleotide} (Expected: GT)")
print(f"3' Acceptor Site  : {acceptor_dinucleotide} (Expected: AG)")
print(f"Is Canonical GT-AG: {is_canonical}")
assert is_canonical is True


---

### 1C.2 Open Reading Frame (ORF) Extraction

#### 4.2.1 Theory: The 6 Reading Frames of Double-Stranded DNA
An **Open Reading Frame (ORF)** starts with `ATG` and continues in frame until terminating at `TAA`, `TAG`, or `TGA`.
Double-stranded DNA yields 6 frames: 3 forward (+1, +2, +3) and 3 reverse (-1, -2, -3).


In [ ]:
CODON_TABLE = {
    'ATA':'I', 'ATC':'I', 'ATT':'I', 'ATG':'M',
    'ACA':'T', 'ACC':'T', 'ACG':'T', 'ACT':'T',
    'AAC':'N', 'AAT':'N', 'AAA':'K', 'AAG':'K',
    'AGC':'S', 'AGT':'S', 'AGA':'R', 'AGG':'R',
    'CTA':'L', 'CTC':'L', 'CTG':'L', 'CTT':'L',
    'CCA':'P', 'CCC':'P', 'CCG':'P', 'CCT':'P',
    'CAC':'H', 'CAT':'H', 'CAA':'Q', 'CAG':'Q',
    'CGA':'R', 'CGC':'R', 'CGG':'R', 'CGT':'R',
    'GTA':'V', 'GTC':'V', 'GTG':'V', 'GTT':'V',
    'GCA':'A', 'GCC':'A', 'GCG':'A', 'GCT':'A',
    'GAC':'D', 'GAT':'D', 'GAA':'E', 'GAG':'E',
    'GGA':'G', 'GGC':'G', 'GGG':'G', 'GGT':'G',
    'TCA':'S', 'TCC':'S', 'TCG':'S', 'TCT':'S',
    'TTC':'F', 'TTT':'F', 'TTA':'L', 'TTG':'L',
    'TAC':'Y', 'TAT':'Y', 'TAA':'*', 'TAG':'*',
    'TGC':'C', 'TGT':'C', 'TGA':'*', 'TGG':'W',
}

def find_orfs_single_frame(seq: str, frame_offset: int = 0, min_aa: int = 5) -> list[dict]:
    clean = seq.upper()
    stop_codons = {"TAA", "TAG", "TGA"}
    orfs = []
    i = frame_offset
    while i < len(clean) - 2:
        if clean[i:i+3] == "ATG":
            start_pos = i
            for j in range(start_pos + 3, len(clean) - 2, 3):
                if clean[j:j+3] in stop_codons:
                    end_pos = j + 3
                    orf_nt = clean[start_pos:end_pos]
                    protein = "".join(CODON_TABLE.get(orf_nt[k:k+3], 'X') for k in range(0, len(orf_nt) - 3, 3))
                    if len(protein) >= min_aa:
                        orfs.append({"frame": frame_offset, "start": start_pos, "end": end_pos, "length_aa": len(protein), "protein": protein})
                    break
        i += 3
    return orfs

test_seq = "CCATGGCCATTGTAATGGGCCGCTGAAAGGGT"
detected_orfs = find_orfs_single_frame(test_seq, frame_offset=2, min_aa=5)
assert len(detected_orfs) == 1
assert detected_orfs[0]["protein"] == "MAIVMGR"


---

### 1C.3 Exon Extraction & Transcript Assembly

#### 4.3.1 Theory: Splicing Exons into Mature Coding Sequences
To reconstruct mature coding sequence (CDS), slice genomic DNA using coordinate intervals `[start, end)` and join them.


In [ ]:
toy_genomic_dna = "ATGGGACGCT" + "GTAAGTAG" + "GCTAGCTA" + "GTACGTACAG" + "CGTACTAGTAA"
exon_coordinates = [(0, 10), (18, 26), (36, 47)]

def assemble_transcript(genomic_seq: str, exon_coords: list[tuple[int, int]]) -> str:
    exons = [genomic_seq[start:end] for start, end in exon_coords]
    return "".join(exons)

mature_cds = assemble_transcript(toy_genomic_dna, exon_coordinates)
print("Assembled CDS:", mature_cds)
assert mature_cds.startswith("ATG")
assert mature_cds.endswith("TAA")


---

### 1C.4 Intron Extraction & Splice Site Verification

#### 4.4.1 Theory: Inter-Exonic Intervals
Introns correspond strictly to genomic intervals *between* consecutive exons: $\text{Intron}_i = \text{Genomic}[E_i : S_{i+1}]$.


In [ ]:
def extract_and_audit_introns(genomic_seq: str, exon_coords: list[tuple[int, int]]) -> list[dict]:
    introns = []
    for i in range(len(exon_coords) - 1):
        intron_start = exon_coords[i][1]
        intron_end = exon_coords[i+1][0]
        intron_seq = genomic_seq[intron_start:intron_end]
        donor = intron_seq[:2]
        acceptor = intron_seq[-2:]
        introns.append({
            "intron_index": i + 1, "start": intron_start, "end": intron_end,
            "length": len(intron_seq), "donor": donor, "acceptor": acceptor,
            "is_canonical_gt_ag": (donor == "GT" and acceptor == "AG")
        })
    return introns

intron_audit = extract_and_audit_introns(toy_genomic_dna, exon_coordinates)
for intr in intron_audit:
    assert intr['is_canonical_gt_ag'] is True


---

### 1C.5 Coordinate Systems: 0-Based Half-Open vs. 1-Based Closed

#### 4.5.1 Theory: The #1 Source of Bioinformatics Bugs
- **0-based half-open** `[start, end)` (Python, BED): first element index 0, length = $end - start$.
- **1-based closed** `[start, end]` (GenBank, GFF, GTF, VCF): first coordinate 1, length = $end - start + 1$.
- Conversion: $S_0 = S_1 - 1, E_0 = E_1$.


In [ ]:
def gff_to_python_slice(start_1based: int, end_1based: int) -> tuple[int, int]:
    return (start_1based - 1, end_1based)

def python_slice_to_gff(start_0based: int, end_0based: int) -> tuple[int, int]:
    return (start_0based + 1, end_0based)

s0, e0 = gff_to_python_slice(1, 3)
assert (s0, e0) == (0, 3)
assert python_slice_to_gff(0, 3) == (1, 3)


## 5. Practice Exercises

Implement the 10 exercises below.


### Exercise 1 (Level 1 ★): GFF Coordinate Converter and Slice Extractor

**Problem**: Write a function `extract_from_gff_coords(sequence: str, start_1based: int, end_1based: int) -> str` converting 1-based interval to Python slice.

- **Input**: Sequence string and 1-based start/end.
- **Expected Output**: Extracted string.


In [ ]:
def extract_from_gff_coords(sequence: str, start_1based: int, end_1based: int) -> str:
    return sequence[start_1based - 1 : end_1based]


#### 💡 Solution Explanation
Subtracts 1 from 1-based start; end coordinate remains unchanged.

In [ ]:
dna = 'ATGCGTACTGAC'
assert extract_from_gff_coords(dna, 1, 3) == 'ATG'
assert extract_from_gff_coords(dna, 4, 6) == 'CGT'
print('✓ Exercise 1 passed!')


### Exercise 2 (Level 1 ★): Forward Single-Frame ORF Extractor

**Problem**: Write a function `extract_single_frame_orfs(seq: str, min_aa: int = 5) -> list[str]` returning translated peptides from frame 0.

- **Input**: DNA string `seq`, integer `min_aa`.
- **Expected Output**: List of peptide strings.


In [ ]:
def extract_single_frame_orfs(seq: str, min_aa: int = 5) -> list[str]:
    clean = seq.upper()
    stops = {"TAA", "TAG", "TGA"}
    peptides = []
    i = 0
    while i < len(clean) - 2:
        if clean[i:i+3] == "ATG":
            start_pos = i
            for j in range(start_pos + 3, len(clean) - 2, 3):
                if clean[j:j+3] in stops:
                    orf_nt = clean[start_pos:j]
                    prot = "".join(CODON_TABLE.get(orf_nt[k:k+3], 'X') for k in range(0, len(orf_nt), 3))
                    if len(prot) >= min_aa:
                        peptides.append(prot)
                    break
        i += 3
    return peptides


#### 💡 Solution Explanation
Finds ATG, scans in-frame for stop, translates, filters by min_aa.

In [ ]:
assert extract_single_frame_orfs('ATGGCCATTGTAATGGGCTGA', min_aa=5) == ['MAIVMG']
print('✓ Exercise 2 passed!')


### Exercise 3 (Level 2 ★★): Exon Assembler and Transcript Auditor

**Problem**: Write a function `assemble_and_verify_cds(genomic_seq: str, exon_coords: list[tuple[int, int]]) -> dict` assembling exons and auditing triplet length, start codon, and stop codon.

- **Input**: Genomic sequence and coordinate list.
- **Expected Output**: Audit dictionary.


In [ ]:
def assemble_and_verify_cds(genomic_seq: str, exon_coords: list[tuple[int, int]]) -> dict:
    cds = "".join(genomic_seq[s:e] for s, e in exon_coords).upper()
    stops = {"TAA", "TAG", "TGA"}
    return {
        "cds": cds,
        "length": len(cds),
        "is_triplet": (len(cds) % 3) == 0,
        "starts_atg": cds.startswith("ATG"),
        "ends_stop": cds[-3:] in stops if len(cds) >= 3 else False
    }


#### 💡 Solution Explanation
Slices and concatenates exonic regions and performs structural validation.

In [ ]:
res = assemble_and_verify_cds('NNNATGGCCNNNTAG', [(3, 9), (12, 15)])
assert res['starts_atg'] is True
assert res['ends_stop'] is True
print('✓ Exercise 3 passed!')


### Exercise 4 (Level 2 ★★): Intron Boundary Extractor & GT-AG Auditor

**Problem**: Write a function `audit_intron_splice_sites(genomic_seq: str, exon_coords: list[tuple[int, int]]) -> list[dict]` checking canonical GT-AG signals on all introns.

- **Input**: Genomic DNA and coordinate list.
- **Expected Output**: List of intron dictionaries.


In [ ]:
def audit_intron_splice_sites(genomic_seq: str, exon_coords: list[tuple[int, int]]) -> list[dict]:
    clean = genomic_seq.upper()
    introns = []
    for i in range(len(exon_coords) - 1):
        s = exon_coords[i][1]
        e = exon_coords[i+1][0]
        seq = clean[s:e]
        d = seq[:2] if len(seq) >= 2 else ""
        a = seq[-2:] if len(seq) >= 2 else ""
        introns.append({"index": i + 1, "start": s, "end": e, "donor": d, "acceptor": a, "is_canonical": (d == "GT" and a == "AG")})
    return introns


#### 💡 Solution Explanation
Extracts sequences between exons and validates splice donor and acceptor dinucleotides.

In [ ]:
audit = audit_intron_splice_sites('EXON1GTAAGEXON2GTCCAGEXON3', [(0, 5), (10, 15), (21, 26)])
assert len(audit) == 2
assert audit[0]['is_canonical'] is True
print('✓ Exercise 4 passed!')


### Exercise 5 (Level 3 ★★★): End-to-End Gene Model Parser on Real Data

**Problem**: Write a function `parse_real_gene_model(gene_file_path: Path, coords_file_path: Path) -> dict` parsing real files from `data/sequences/toy_gene_with_introns.txt` and `data/tables/toy_exon_coordinates.tsv`.

- **Input**: Paths to gene file and TSV file.
- **Expected Output**: Summary dictionary.


In [ ]:
def parse_real_gene_model(gene_file_path: Path, coords_file_path: Path) -> dict:
    with open(gene_file_path, "r") as f:
        gene_seq = f.read().strip().upper()
    exon_coords = []
    with open(coords_file_path, "r") as f:
        lines = [line.strip().split("\t") for line in f if line.strip()]
        s_idx, e_idx = lines[0].index("start"), lines[0].index("end")
        for row in lines[1:]:
            exon_coords.append((int(row[s_idx]), int(row[e_idx])))
    cds = "".join(gene_seq[s:e] for s, e in exon_coords)
    prot = []
    for k in range(0, len(cds) - 2, 3):
        aa = CODON_TABLE.get(cds[k:k+3], "X")
        if aa == "*": break
        prot.append(aa)
    all_canon = True
    for i in range(len(exon_coords) - 1):
        iseq = gene_seq[exon_coords[i][1] : exon_coords[i+1][0]]
        if not (iseq.startswith("GT") and iseq.endswith("AG")):
            all_canon = False
            break
    return {"cds_sequence": cds, "protein_sequence": "".join(prot), "num_exons": len(exon_coords), "num_introns": len(exon_coords) - 1, "all_introns_canonical": all_canon}


#### 💡 Solution Explanation
Parses real coordinate files, splices exons into CDS, translates to protein, and validates all intron junctions.

In [ ]:
gene_f = DATA_DIR / "sequences" / "toy_gene_with_introns.txt"
tsv_f = DATA_DIR / "tables" / "toy_exon_coordinates.tsv"
res = parse_real_gene_model(gene_f, tsv_f)
assert res["num_exons"] == 3
assert res["all_introns_canonical"] is True
print('✓ Exercise 5 passed!')


### Exercise 6 (Level 1 ★): BED to GenBank Coordinate Converter

**Problem**: Write a function `bed_to_genbank_coords(start_0based: int, end_0based: int) -> tuple[int, int]` converting 0-based half-open interval to 1-based closed interval.

- **Input**: Integers `start_0based` and `end_0based`.
- **Expected Output**: Tuple `(start_1based, end_1based)`.


In [ ]:
def bed_to_genbank_coords(start_0based: int, end_0based: int) -> tuple[int, int]:
    return (start_0based + 1, end_0based)


#### 💡 Solution Explanation
Converts BED interval $[s, e)$ to GenBank $[s+1, e]$.

In [ ]:
assert bed_to_genbank_coords(0, 100) == (1, 100)
print('✓ Exercise 6 passed!')


### Exercise 7 (Level 2 ★★): Alternative Splicing Exon Skipper

**Problem**: Write a function `assemble_splice_isoforms(genomic: str, constitutive_exons: list[tuple[int, int]], cassette_exon: tuple[int, int]) -> dict` assembling both Isoform A (including cassette exon inserted between constitutive exons) and Isoform B (excluding cassette exon). Return `{'isoform_included': str, 'isoform_skipped': str}`.

- **Input**: Genomic sequence, list of 2 constitutive exon tuples, 1 cassette exon tuple.
- **Expected Output**: Dictionary with both isoforms.


In [ ]:
def assemble_splice_isoforms(genomic: str, constitutive_exons: list[tuple[int, int]], cassette_exon: tuple[int, int]) -> dict:
    e1 = genomic[constitutive_exons[0][0] : constitutive_exons[0][1]]
    e2 = genomic[constitutive_exons[1][0] : constitutive_exons[1][1]]
    ecass = genomic[cassette_exon[0] : cassette_exon[1]]
    return {
        'isoform_included': (e1 + ecass + e2).upper(),
        'isoform_skipped': (e1 + e2).upper()
    }


#### 💡 Solution Explanation
Models cassette exon alternative splicing, which generates proteomic diversity from a single gene locus.

In [ ]:
genomic = "AAA" + "NN" + "CCC" + "NN" + "GGG"
res = assemble_splice_isoforms(genomic, [(0, 3), (10, 13)], (5, 8))
assert res['isoform_included'] == "AAACCCGGG"
assert res['isoform_skipped'] == "AAAGGG"
print('✓ Exercise 7 passed!')


### Exercise 8 (Level 2 ★★): Splice Donor Consensus Matcher

**Problem**: Write a function `is_strong_donor_site(splice_seq: str) -> bool` testing if a 6-bp splice donor sequence conforms to the mammalian consensus `[CA]AG|GT[AG]AG`. Return `True` if it starts with GT and matches consensus positions, else `False`.

- **Input**: 6-bp candidate donor string (first 2 bp must be GT).
- **Expected Output**: Boolean.


In [ ]:
def is_strong_donor_site(splice_seq: str) -> bool:
    clean = splice_seq.upper()
    if len(clean) != 6: return False
    # Standard consensus donor: G T [A/G] A G [T/C/A]
    return clean[:2] == "GT" and clean[2] in "AG" and clean[3] == "A" and clean[4] == "G"


#### 💡 Solution Explanation
Evaluates splice donor sequence against consensus frequency motif.

In [ ]:
assert is_strong_donor_site('GTAAGT') is True
assert is_strong_donor_site('GTGAGC') is True
assert is_strong_donor_site('ATAAAA') is False
print('✓ Exercise 8 passed!')


### Exercise 9 (Level 3 ★★★): 6-Frame Genomic ORF Finder

**Problem**: Write a function `find_6frame_genomic_orfs(dna_seq: str, min_aa: int = 10) -> list[dict]` searching all 6 reading frames and returning detected ORFs sorted by peptide length descending.

- **Input**: DNA sequence string and integer `min_aa`.
- **Expected Output**: List of ORF dictionaries.


In [ ]:
def find_6frame_genomic_orfs(dna_seq: str, min_aa: int = 10) -> list[dict]:
    clean = dna_seq.upper()
    _rev_t = str.maketrans("ACGT", "TGCA")
    rc = clean.translate(_rev_t)[::-1]
    stops = {"TAA", "TAG", "TGA"}
    results = []
    
    def scan(strand_seq, strand_sign):
        for frame in (0, 1, 2):
            i = frame
            while i < len(strand_seq) - 2:
                if strand_seq[i:i+3] == "ATG":
                    start = i
                    for j in range(start + 3, len(strand_seq) - 2, 3):
                        if strand_seq[j:j+3] in stops:
                            orf_nt = strand_seq[start:j]
                            prot = "".join(CODON_TABLE.get(orf_nt[k:k+3], 'X') for k in range(0, len(orf_nt), 3))
                            if len(prot) >= min_aa:
                                results.append({
                                    "strand": strand_sign,
                                    "frame": frame + 1,
                                    "length_aa": len(prot),
                                    "protein": prot
                                })
                            break
                i += 3
                
    scan(clean, "+")
    scan(rc, "-")
    return sorted(results, key=lambda x: -x["length_aa"])


#### 💡 Solution Explanation
Systematic 6-frame scan translating forward and reverse complement strands.

In [ ]:
seq = "ATG" + ("GCC" * 12) + "TAA"
orfs = find_6frame_genomic_orfs(seq, min_aa=10)
assert len(orfs) >= 1
assert orfs[0]["length_aa"] == 13
print('✓ Exercise 9 passed!')


### Exercise 10 (Level 3 ★★★): Transcript-to-Genomic Coordinate Mapper

**Problem**: Write a function `transcript_to_genomic_coord(transcript_pos: int, exon_coords: list[tuple[int, int]]) -> int` mapping a 0-based coordinate on the spliced mature mRNA back to its corresponding 0-based genomic coordinate across multiple exons. If `transcript_pos` exceeds the total spliced length, raise a `ValueError`.

- **Input**: Integer `transcript_pos` and list of sorted 0-based exon intervals `[(s, e), ...]`. 
- **Expected Output**: Integer genomic coordinate.


In [ ]:
def transcript_to_genomic_coord(transcript_pos: int, exon_coords: list[tuple[int, int]]) -> int:
    if transcript_pos < 0: raise ValueError("Negative transcript coordinate!")
    running_offset = 0
    for s, e in exon_coords:
        exon_len = e - s
        if running_offset <= transcript_pos < running_offset + exon_len:
            offset_within_exon = transcript_pos - running_offset
            return s + offset_within_exon
        running_offset += exon_len
    raise ValueError("Coordinate exceeds transcript length!")


#### 💡 Solution Explanation
Iterates through exon spans, tracking cumulative spliced offset to pinpoint the exact genomic chromosome coordinate.

In [ ]:
# Exon 1: [100, 150) (len 50) -> transcript coords 0..49
# Exon 2: [300, 380) (len 80) -> transcript coords 50..129
exons = [(100, 150), (300, 380)]
assert transcript_to_genomic_coord(0, exons) == 100
assert transcript_to_genomic_coord(49, exons) == 149
assert transcript_to_genomic_coord(50, exons) == 300
assert transcript_to_genomic_coord(55, exons) == 305
print('✓ Exercise 10 passed!')


## 6. Summary & Cheat Sheet Table

| Concept | Biology Rule | Python Expression / Algorithm |
|---|---|---|
| **ORF Definition** | `ATG` $\dots$ in-frame Stop (`TAA`/`TAG`/`TGA`) | `while` loop scanning 3-bp windows per frame |
| **6 Reading Frames** | 3 Forward (+1, +2, +3), 3 Reverse (-1, -2, -3) | Forward frames + frames on `reverse_complement(seq)` |
| **Exon Splicing** | Concatenate exonic segments | `"".join(genomic[s:e] for s, e in exon_coords)` |
| **Intron Boundaries** | Inter-exonic intervals | `genomic[exon[i][1] : exon[i+1][0]]` |
| **Splice Consensus** | Canonical 5' Donor `GT` / 3' Acceptor `AG` | `intron.startswith("GT") and intron.endswith("AG")` |
| **1-based to 0-based** | GFF/GenBank closed $[S, E]$ to Python $[s, e)$ | `start_0 = S - 1`, `end_0 = E` |
| **0-based to 1-based** | Python slice $[s, e)$ to GFF closed $[S, E]$ | `start_1 = s + 1`, `end_1 = e` |


## 7. Further Reading & References

1. **Mount, S. M. (1982)**. A catalogue of splice junction sequences. *Nucleic Acids Research*, 10(2), 459-472.
2. **Burge, C., & Karlin, S. (1997)**. Prediction of complete gene structures in human genomic DNA. *Journal of Molecular Biology*, 268(1), 78-94.
3. **UCSC Genome Browser FAQ**: *Coordinate Conventions: 0-Based vs. 1-Based*. https://genome.ucsc.edu/FAQ/FAQtracks.html#tracks1
